In [1]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import pandas as pd
from sklearn.cluster import HDBSCAN
import umap
from tqdm import tqdm

/home/aru/Work/shome2023notebook/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
asserts = pd.read_csv(
    "../data/asserts.csv",
    header=None,
    names=["notebook", "assert"]
)
asserts

,notebook,assert
32,data/assert_notebooks/rohitashwachaks/MIS-382N...,"assert train_loss < 0.5, train_loss"
32,data/assert_notebooks/rohitashwachaks/MIS-382N...,"assert train_acc <= 1 and train_acc > 0.7, tra..."
32,data/assert_notebooks/rohitashwachaks/MIS-382N...,"assert test_acc <= 1 and test_acc > 0.7, test_acc"
32,data/assert_notebooks/rohitashwachaks/MIS-382N...,"assert train_loss < 0.5, train_loss"
32,data/assert_notebooks/rohitashwachaks/MIS-382N...,"assert train_acc <= 1 and train_acc > 0.7, tra..."
...,...,...
6,data/quaranta2021kgtorrent/KT_dataset/ashishpa...,assert shape[2] == 3
16,data/quaranta2021kgtorrent/KT_dataset/aryansha...,assert type(self.targetName) == str
16,data/quaranta2021kgtorrent/KT_dataset/aryansha...,assert type(self.colnames) == str
16,data/quaranta2021kgtorrent/KT_dataset/aryansha...,assert self.colnames in X.columns


In [14]:
statements = asserts.sample(frac=0.01)
statements

,notebook,assert
4,data/assert_notebooks/TapasKumarDutta1/Multihe...,assert len(a) == len(b)
74,data/assert_notebooks/lrthomps/cs224u/hw_wordr...,"assert np.array_equal(result, expected), 'Your..."
6,data/assert_notebooks/samstikhin/ml2021/09-NLP...,"assert np.allclose(tf, tf_r, atol=0.01)"
28,data/assert_notebooks/Hari31416/Data-Sciences/...,"assert np.isclose(scores[2].numpy(), 171.60194..."
26,data/assert_notebooks/pwais/tpu/tools/colab/ke...,"assert PROJECT, 'For this part, you need a GCP..."
...,...,...
24,data/assert_notebooks/nikita-mishunyayev/Kaggl...,assert len(pp_mult) == len(sub_test)
19,data/quaranta2021kgtorrent/KT_dataset/hpant438...,assert len(x) == 2
2,data/assert_notebooks/lukas-krecan/ml_examples...,assert self.trained == True
29,data/assert_notebooks/WAlbers99/MinorProject/K...,"assert helpful_eq(new_X_reconstr, np.array([[-..."


In [ ]:
MODEL = "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModel.from_pretrained(MODEL)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [16]:
def embed_batch(texts: list[str], batch_size: int = 32) -> np.ndarray:
    """Return CLS-token embeddings, shape (n, 768)."""
    all_vecs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch = texts[i : i + batch_size]
        enc = tokenizer (
            batch,
            padding=True,
            truncation=True,
            max_length=128, # assert statements are short
            return_tensors="pt",
        ).to(device)
        with torch.no_grad():
            out = model(**enc)
        # CLS token = out.last_hidden_state[:, 0, :]
        vecs = out.last_hidden_state[:, 0, :].cpu().numpy()
        all_vecs.append(vecs)
    return np.vstack(all_vecs)

X = embed_batch(statements.loc[:, "assert"].to_list()) # (n_samples, 768)

Embedding: 100%|██████████| 28/28 [00:35<00:00,  1.26s/it]


In [17]:
reducer = umap.UMAP(
    n_components=10,
    n_neighbors=15,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
)
X_reduced = reducer.fit_transform(X)

/home/aru/Work/shome2023notebook/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [19]:
clusterer = HDBSCAN(
    min_cluster_size=10,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom",
)
labels = clusterer.fit_predict(X_reduced)
statements["cluster"] = labels
statements

/home/aru/Work/shome2023notebook/.venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


,notebook,assert,cluster
4,data/assert_notebooks/TapasKumarDutta1/Multihe...,assert len(a) == len(b),3
74,data/assert_notebooks/lrthomps/cs224u/hw_wordr...,"assert np.array_equal(result, expected), 'Your...",2
6,data/assert_notebooks/samstikhin/ml2021/09-NLP...,"assert np.allclose(tf, tf_r, atol=0.01)",5
28,data/assert_notebooks/Hari31416/Data-Sciences/...,"assert np.isclose(scores[2].numpy(), 171.60194...",13
26,data/assert_notebooks/pwais/tpu/tools/colab/ke...,"assert PROJECT, 'For this part, you need a GCP...",13
...,...,...,...
24,data/assert_notebooks/nikita-mishunyayev/Kaggl...,assert len(pp_mult) == len(sub_test),17
19,data/quaranta2021kgtorrent/KT_dataset/hpant438...,assert len(x) == 2,12
2,data/assert_notebooks/lukas-krecan/ml_examples...,assert self.trained == True,25
29,data/assert_notebooks/WAlbers99/MinorProject/K...,"assert helpful_eq(new_X_reconstr, np.array([[-...",21


In [26]:
statements["cluster"].value_counts().sort_index()

cluster
-1     94
 0     19
 1     13
 2     20
 3     14
 4     45
 5     40
 6     18
 7     21
 8     15
 9     12
 10    12
 11    18
 12    47
 13    68
 14    20
 15    96
 16    11
 17    15
 18    15
 19    17
 20    14
 21    21
 22    16
 23    41
 24    25
 25    45
 26    10
 27    13
 28    36
 29    29
 30    16
Name: count, dtype: int64

In [27]:
statements.loc[statements["cluster"] == -1].sample(n=5)

,notebook,assert,cluster
4,data/assert_notebooks/sorrge/tg_news_cluster/m...,"assert idx2 not in idx_group[idx1], f'{idx1} a...",-1
38,data/assert_notebooks/tomatiks/education/Pract...,"assert isinstance(loss, Variable) and tuple(lo...",-1
22,data/quaranta2021kgtorrent/KT_dataset/metathes...,assert img is not None,-1
8,data/assert_notebooks/ece1786-2022/CaptionMe/a...,assert mask is not None,-1
12,data/assert_notebooks/rajatguptakgp/practical_...,assert len(X_KF) == len(X_GT) == len(X_MODEL) ...,-1
